# Lab 2 · The harness
MICAI 2026 · Reliable Enterprise Agents · about 40 minutes

**What you will learn**
- Why a lost response makes a naive agent write twice
- Idempotency keys, timeouts, and retries that are safe
- Circuit breakers and compensating actions
- Keeping the context small as a task grows
- Cost per task, and logs you can audit and replay
- Why the harness, not the model, is what keeps data correct under failure
- Harness engineering: tuning a knob by measuring cost *and* failures, not by taste

**What we build together**
- `execute`: the function every tool call goes through
- an agent that logs every step and can be replayed without the model

Run each cell when we reach it on the screen. Cells that start with *Before you run it* ask you to make a prediction first.

## Setup

In [1]:
import json
import os
import random
import time
import uuid
import hashlib
from pathlib import Path

# Locally (uv) everything is already installed. In Colab we install the
# packages and download the data files from the course repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    import urllib.request
    subprocess.run(["pip", "install", "-q", "openai==3.41.0"], check=True)
    Path("data").mkdir(exist_ok=True)
    repo = "https://raw.githubusercontent.com/RamsesCamas/Enterprise-Agents-MICAI-2026/main/labs/data/"
    for name in ["products", "customers", "invoices", "documents"]:
        if not Path(f"data/{name}.jsonl").exists():
            urllib.request.urlretrieve(repo + f"{name}.jsonl", f"data/{name}.jsonl")

print("Data files:", sorted(p.name for p in Path("data").glob("*.jsonl")))

Data files: ['customers.jsonl', 'documents.jsonl', 'invoices.jsonl', 'products.jsonl']


In [2]:
from openai import OpenAI


def _load_dotenv():
    """Read a .env file into os.environ when running locally (not in Colab).

    Looks for a .env next to the notebook and in parent folders. Values
    already set in the real environment win, so `export GROQ_API_KEY=...`
    still takes precedence over the file.
    """
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                os.environ.setdefault(key, value)
            break


def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    # Locally: use the real environment first, then fall back to a .env file.
    if name not in os.environ:
        _load_dotenv()
    return os.environ.get(name)


client = OpenAI(
    base_url=os.environ.get("LLM_BASE_URL", "https://api.groq.com/openai/v1"),
    api_key=get_secret("GROQ_API_KEY") or "missing-key",
    max_retries=4,
)
MODEL_NAME = os.environ.get("LLM_MODEL", "openai/gpt-oss-120b")


def ask_model(messages, tools=None):
    """Send the conversation to the model. Return its reply as a plain dict."""
    if tools:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools, temperature=0)
    else:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, temperature=0)

    message = response.choices[0].message
    reply = {"role": "assistant", "content": message.content}
    if message.tool_calls:
        reply["tool_calls"] = []
        for call in message.tool_calls:
            reply["tool_calls"].append({
                "id": call.id,
                "type": "function",
                "function": {"name": call.function.name, "arguments": call.function.arguments},
            })
    reply["tokens"] = response.usage.total_tokens
    return reply

In [3]:
def read_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


class ERPError(Exception):
    def __init__(self, code, message, retryable=False, retry_after=None):
        super().__init__(message)
        self.code = code
        self.message = message
        self.retryable = retryable
        self.retry_after = retry_after


class AcmeERP:
    """A small in-memory ERP loaded from data/*.jsonl.

    It behaves like a real backend: errors with codes, idempotency keys on
    writes, and outages you can simulate with fail_next().
    """

    def __init__(self, data_dir="data"):
        self.products = {}
        for row in read_jsonl(f"{data_dir}/products.jsonl"):
            self.products[row["sku"]] = row
        self.customers = {}
        for row in read_jsonl(f"{data_dir}/customers.jsonl"):
            self.customers[row["id"]] = row
        self.invoices = {}
        for row in read_jsonl(f"{data_dir}/invoices.jsonl"):
            self.invoices[row["id"]] = row
        self.movements = []
        self.refunds = []
        self.saved_replies = {}      # idempotency key -> reply already sent
        self.pending_failures = []

    def fail_next(self, kind, times=1):
        """kind: 'unavailable', 'rate_limited', 'timeout' or 'lost_response'."""
        for _ in range(times):
            self.pending_failures.append(kind)

    def _check_failures(self):
        """Raise a simulated outage. Return True if the reply must be 'lost'."""
        if not self.pending_failures:
            return False
        kind = self.pending_failures.pop(0)
        if kind == "unavailable":
            raise ERPError("unavailable", "503 Service unavailable", retryable=True, retry_after=1.0)
        if kind == "rate_limited":
            raise ERPError("rate_limited", "429 Too many requests", retryable=True, retry_after=2.0)
        if kind == "timeout":
            raise ERPError("timeout", "No answer after 5 seconds", retryable=True)
        return kind == "lost_response"

    # ---------- reads ----------

    def get_product(self, sku):
        self._check_failures()
        if sku not in self.products:
            raise ERPError("not_found", f"Product {sku} not found")
        return dict(self.products[sku])

    def find_products(self, category=None, max_stock=None):
        self._check_failures()
        found = []
        for product in self.products.values():
            if category is not None and product["category"] != category:
                continue
            if max_stock is not None and product["stock"] > max_stock:
                continue
            found.append(dict(product))
        return found

    def get_customer(self, customer_id):
        self._check_failures()
        if customer_id not in self.customers:
            raise ERPError("not_found", f"Customer {customer_id} not found")
        return dict(self.customers[customer_id])

    def get_invoice(self, invoice_id):
        self._check_failures()
        if invoice_id not in self.invoices:
            raise ERPError("not_found", f"Invoice {invoice_id} not found")
        return dict(self.invoices[invoice_id])

    # ---------- writes: applied once per idempotency key ----------

    def adjust_stock(self, sku, delta, reason, idempotency_key=None):
        lose_reply = self._check_failures()
        if idempotency_key in self.saved_replies:
            return {**self.saved_replies[idempotency_key], "replayed": True}

        if sku not in self.products:
            raise ERPError("not_found", f"Product {sku} not found")
        product = self.products[sku]
        if product["stock"] + delta < 0:
            raise ERPError("insufficient_stock", f"Stock is {product['stock']}, cannot apply {delta}")
        product["stock"] += delta
        self.movements.append({"sku": sku, "delta": delta, "reason": reason})
        reply = {"sku": sku, "stock": product["stock"]}

        if idempotency_key:
            self.saved_replies[idempotency_key] = reply
        if lose_reply:
            # The write is committed, but the client never gets the answer.
            raise ERPError("bad_gateway", "502 Connection reset. The write may have been applied.", retryable=True)
        return reply

    def create_refund(self, invoice_id, amount, reason, idempotency_key=None):
        lose_reply = self._check_failures()
        if idempotency_key in self.saved_replies:
            return {**self.saved_replies[idempotency_key], "replayed": True}

        if invoice_id not in self.invoices:
            raise ERPError("not_found", f"Invoice {invoice_id} not found")
        invoice = self.invoices[invoice_id]
        if invoice["status"] != "paid":
            raise ERPError("invalid_state", "Only paid invoices can be refunded")
        available = invoice["total"] - invoice["refunded"]
        if amount > available:
            raise ERPError("exceeds_refundable", f"Refundable balance is {available}")
        invoice["refunded"] += amount
        self.refunds.append({"invoice_id": invoice_id, "amount": amount, "reason": reason})
        reply = {"invoice_id": invoice_id, "refunded": amount, "remaining": available - amount}

        if idempotency_key:
            self.saved_replies[idempotency_key] = reply
        if lose_reply:
            raise ERPError("bad_gateway", "502 Connection reset. The write may have been applied.", retryable=True)
        return reply


erp = AcmeERP()
print(len(erp.products), "products,", len(erp.customers), "customers,", len(erp.invoices), "invoices")

12 products, 8 customers, 6 invoices


In [4]:
class FakeModel:
    """Plays back replies we prepared. Lets us test without network or tokens."""

    def __init__(self, replies):
        self.replies = list(replies)

    def __call__(self, messages, tools=None):
        return dict(self.replies.pop(0))


def tool_reply(name, **arguments):
    """A prepared reply in which the model asks for one tool call."""
    call = {"id": "call_" + uuid.uuid4().hex[:6], "type": "function",
            "function": {"name": name, "arguments": json.dumps(arguments)}}
    return {"role": "assistant", "content": None, "tool_calls": [call], "tokens": 300}


def text_reply(text):
    return {"role": "assistant", "content": text, "tokens": 300}

## 1. See it fail
`lost_response` means the ERP saves the change, then the connection drops and the client receives a 502.
Seeing an error, the agent does what a person would do: it tries again.

*Before you run it:* after two attempts of "+3 units", how many stock movements will the ERP have?

In [5]:
erp = AcmeERP()
erp.fail_next("lost_response")

for attempt in [1, 2]:
    try:
        print(attempt, erp.adjust_stock("SKU-001", 3, "restock"))
    except ERPError as error:
        print(attempt, "error:", error.message)

print("stock movements:", len(erp.movements), "(we wanted 1)")

1 error: 502 Connection reset. The write may have been applied.
2 {'sku': 'SKU-001', 'stock': 63}
stock movements: 2 (we wanted 1)


## 2. One function for every tool call
`execute` never raises. It returns `{"ok": ...}` or `{"error": ...}`.

1. **Writes carry an idempotency key.** The key is computed from the task and the arguments, so a retry sends the same key and the ERP returns the saved reply instead of writing again.
2. **Some errors must never be retried:** not found, forbidden, not enough stock. They go back to the model with a hint.
3. **Retry only when it is safe:** reads, or writes that carry a key. Wait the time the server asks for (`Retry-After`), otherwise wait a little longer each time.

In [6]:
HINTS = {
    "not_found": "Check the identifier, or search first instead of guessing.",
    "insufficient_stock": "Tell the user how much stock there is. Do not retry.",
    "forbidden": "You do not have permission. Do not retry. Tell the user.",
}


def make_idempotency_key(task_id, tool_name, arguments):
    text = json.dumps({"task": task_id, "tool": tool_name, "arguments": arguments}, sort_keys=True)
    return "key_" + hashlib.sha256(text.encode()).hexdigest()[:16]


def execute(tool_name, function, arguments, effect, task_id, max_attempts=3, sleep=time.sleep):
    is_write = effect != "read"
    key = None
    if is_write:
        key = make_idempotency_key(task_id, tool_name, arguments)

    for attempt in range(1, max_attempts + 1):
        try:
            if is_write:
                result = function(**arguments, idempotency_key=key)
            else:
                result = function(**arguments)
            return {"ok": result, "attempts": attempt, "key": key}

        except ERPError as error:
            if not error.retryable:
                return {"error": {"code": error.code, "message": error.message, "hint": HINTS.get(error.code, "")},
                        "attempts": attempt, "key": key}

            safe_to_retry = (not is_write) or (key is not None)
            if attempt == max_attempts or not safe_to_retry:
                return {"error": {"code": error.code, "message": error.message,
                                  "hint": "Retries exhausted. If this was a write, read the record before trying again."},
                        "attempts": attempt, "key": key}

            if error.retry_after is not None:
                wait = error.retry_after
            else:
                wait = 0.5 * (2 ** (attempt - 1)) * random.random()
            sleep(wait)

## 3. Check it

In [7]:
def no_wait(seconds):
    pass


erp = AcmeERP()
erp.fail_next("lost_response")
result = execute("adjust_stock", erp.adjust_stock, {"sku": "SKU-001", "delta": 3, "reason": "restock"},
                 "write", "task_1", sleep=no_wait)
print("lost response: ", result, "| movements:", len(erp.movements))

result = execute("get_product", erp.get_product, {"sku": "SKU-999"}, "read", "task_2")
print("not found:     ", result)

waits = []
erp.fail_next("rate_limited")
result = execute("get_product", erp.get_product, {"sku": "SKU-001"}, "read", "task_3", sleep=waits.append)
print("rate limited:  ", "waited", waits, "then", result["attempts"], "attempts")

erp.fail_next("timeout")
result = execute("get_product", erp.get_product, {"sku": "SKU-001"}, "read", "task_4", sleep=no_wait)
print("timeout:       ", result["attempts"], "attempts,", "ok" if "ok" in result else "failed")

lost response:  {'ok': {'sku': 'SKU-001', 'stock': 60, 'replayed': True}, 'attempts': 2, 'key': 'key_618a1ff28814df72'} | movements: 1
not found:      {'error': {'code': 'not_found', 'message': 'Product SKU-999 not found', 'hint': 'Check the identifier, or search first instead of guessing.'}, 'attempts': 1, 'key': None}
rate limited:   waited [2.0] then 2 attempts
timeout:        2 attempts, ok


## 4. Circuit breaker
When a system keeps failing, retrying only adds load. After a few failures in a row we stop calling it
for a while and tell the model the service is down (OWASP ASI08, cascading failures).

In [9]:
class CircuitBreaker:
    def __init__(self, max_failures=3, cooldown_seconds=15):
        self.max_failures = max_failures
        self.cooldown_seconds = cooldown_seconds
        self.failures = 0
        self.opened_at = None

    def is_open(self):
        if self.opened_at is None:
            return False
        if time.time() - self.opened_at > self.cooldown_seconds:
            self.opened_at = None          # let one call through to test the service
            return False
        return True

    def record(self, result):
        outage_codes = ["unavailable", "rate_limited", "timeout", "bad_gateway"]
        if "error" in result and result["error"]["code"] in outage_codes:
            self.failures += 1
            if self.failures >= self.max_failures:
                self.opened_at = time.time()
        else:
            self.failures = 0


erp = AcmeERP()
erp.fail_next("unavailable", times=20)
breaker = CircuitBreaker(max_failures=3)

for i in range(4):
    if breaker.is_open():
        print(i, "circuit open: we do not even call the ERP")
        continue
    result = execute("get_product", erp.get_product, {"sku": "SKU-001"}, "read", "task", max_attempts=1)
    breaker.record(result)
    print(i, result["error"]["code"])

0 unavailable
1 unavailable
2 unavailable
3 circuit open: we do not even call the ERP


## 5. Compensating actions
A task that writes several times either completes or undoes what it already did, in reverse order.
This is a function the model calls once. The model does not orchestrate the undo.

In [10]:
def receive_shipment(lines, task_id):
    """lines: list of (sku, quantity). All lines are added, or none."""
    done = []
    for sku, quantity in lines:
        result = execute("adjust_stock", erp.adjust_stock,
                         {"sku": sku, "delta": quantity, "reason": "restock"}, "write", task_id)
        if "error" in result:
            for done_sku, done_quantity in reversed(done):
                execute("adjust_stock", erp.adjust_stock,
                        {"sku": done_sku, "delta": -done_quantity, "reason": "correction"}, "write", task_id)
            return {"error": f"Rejected at {sku}. Undid {len(done)} line(s)."}
        done.append((sku, quantity))
    return {"ok": f"Received {len(done)} line(s)."}


erp = AcmeERP()
print(receive_shipment([("SKU-001", 5), ("SKU-002", 5), ("SKU-999", 1)], "shipment_1"))
print("SKU-001 stock:", erp.products["SKU-001"]["stock"])
for movement in erp.movements:
    print("  ", movement)

{'error': 'Rejected at SKU-999. Undid 2 line(s).'}
SKU-001 stock: 57
   {'sku': 'SKU-001', 'delta': 5, 'reason': 'restock'}
   {'sku': 'SKU-002', 'delta': 5, 'reason': 'restock'}
   {'sku': 'SKU-002', 'delta': -5, 'reason': 'correction'}
   {'sku': 'SKU-001', 'delta': -5, 'reason': 'correction'}


## 6. Keeping the context small
Each step sends the whole conversation again, and old tool results are most of it. We shorten all but the
latest ones, but keep every message, so each tool call still has its answer.

Long-term memory is a different problem: anything saved there can affect later tasks, including text an
attacker managed to plant (OWASP ASI06). Keep it per user, reviewable, and do not save raw tool output in it.

In [11]:
def shorten_old_results(messages, keep_last=2, max_chars=120):
    tool_positions = [i for i, m in enumerate(messages) if m["role"] == "tool"]
    old_positions = tool_positions[:-keep_last] if keep_last else tool_positions

    shortened = []
    for i, message in enumerate(messages):
        if i in old_positions and len(message["content"]) > max_chars:
            message = dict(message)
            message["content"] = message["content"][:max_chars] + " ...[shortened]"
        shortened.append(message)
    return shortened


erp = AcmeERP()
conversation = [{"role": "system", "content": "..."}, {"role": "user", "content": "Review the whole catalog"}]
for _ in range(6):
    reply = tool_reply("find_products")
    reply.pop("tokens")
    conversation.append(reply)
    conversation.append({"role": "tool", "tool_call_id": reply["tool_calls"][0]["id"],
                         "content": json.dumps(erp.find_products())})

def approx_tokens(messages):
    return len(json.dumps(messages)) // 4

print("before:", approx_tokens(conversation), "tokens")
print("after: ", approx_tokens(shorten_old_results(conversation)), "tokens")

before: 2431 tokens
after:  1203 tokens


## 7. Putting it together: an agent with a log
The log uses the names from the **OpenTelemetry GenAI conventions** (`invoke_agent`, `chat`, `execute_tool`),
so tools like Phoenix, Langfuse or Datadog can read it. The log holds metadata only.

What the model said goes to a separate **tape**. With the tape we can replay a task later without calling the model.

In [12]:
def adjust_stock(sku, delta, reason, idempotency_key=None):
    return erp.adjust_stock(sku, delta, reason, idempotency_key)


def get_product(sku):
    return erp.get_product(sku)


TOOLS = {
    "get_product": {
        "function": get_product, "effect": "read",
        "schema": {"description": "Get one product: name, price (MXN) and stock (units).",
                   "parameters": {"type": "object", "required": ["sku"],
                                  "properties": {"sku": {"type": "string", "pattern": "^SKU-[0-9]{3}$"}}}},
    },
    "adjust_stock": {
        "function": adjust_stock, "effect": "write",
        "schema": {"description": "Change the stock of one product by delta units (negative to remove).",
                   "parameters": {"type": "object", "required": ["sku", "delta", "reason"],
                                  "properties": {
                                      "sku": {"type": "string", "pattern": "^SKU-[0-9]{3}$"},
                                      "delta": {"type": "integer", "minimum": -1000, "maximum": 1000},
                                      "reason": {"type": "string", "enum": ["restock", "sale", "damage", "correction"]}}}},
    },
}

def tool_specs(tools):
    """What the model sees: name, description and schema. Never the function."""
    specs = []
    for name, tool in tools.items():
        specs.append({"type": "function", "function": {"name": name, **tool["schema"]}})
    return specs


LOG = []      # what happened (audit)
TAPES = {}    # what the model said, per task (for replay)


def log(event, task_id, **details):
    LOG.append({"time": round(time.time(), 3), "gen_ai.operation.name": event,
                "gen_ai.conversation.id": task_id, **details})


def run_agent(model, tools, task, task_id=None, max_steps=6, max_seconds=60):
    task_id = task_id or "task_" + uuid.uuid4().hex[:6]
    TAPES[task_id] = {"task": task, "replies": []}
    messages = [{"role": "system", "content": "You are Acme's operations agent. Use the tools. Only do what is asked."},
                {"role": "user", "content": task}]
    log("invoke_agent", task_id, status="started")
    started = time.time()

    for step in range(1, max_steps + 1):
        if time.time() - started > max_seconds:
            log("invoke_agent", task_id, status="time_budget")
            return None, task_id

        call_started = time.time()
        reply = model(shorten_old_results(messages), tool_specs(tools))
        tokens = reply.pop("tokens", 0)
        TAPES[task_id]["replies"].append(dict(reply))
        log("chat", task_id, step=step, tokens=tokens, ms=round((time.time() - call_started) * 1000))
        messages.append(reply)

        if not reply.get("tool_calls"):
            log("invoke_agent", task_id, status="done")
            return reply["content"], task_id

        for tool_call in reply["tool_calls"]:
            name = tool_call["function"]["name"]
            arguments = json.loads(tool_call["function"]["arguments"] or "{}")
            tool_started = time.time()
            result = execute(name, tools[name]["function"], arguments, tools[name]["effect"], task_id)
            replayed = "ok" in result and isinstance(result["ok"], dict) and result["ok"].get("replayed", False)
            log("execute_tool", task_id, step=step, **{"gen_ai.tool.name": name}, arguments=arguments,
                attempts=result["attempts"], key=result["key"], replayed=replayed,
                status="error" if "error" in result else "ok", ms=round((time.time() - tool_started) * 1000))
            content = result["ok"] if "ok" in result else result["error"]
            messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": json.dumps(content)})

    log("invoke_agent", task_id, status="max_steps")
    return None, task_id

In [13]:
import pandas as pd

erp = AcmeERP()
erp.fail_next("lost_response")
answer, restock_task = run_agent(ask_model, TOOLS, "Restock SKU-012 with 20 units.")
print(answer)
print("stock movements:", erp.movements)

pd.DataFrame([event for event in LOG if event["gen_ai.conversation.id"] == restock_task]).fillna("")

The stock for **SKU‑012** has been increased by **20 units** as a restock. Let me know if you need anything else!
stock movements: [{'sku': 'SKU-012', 'delta': 20, 'reason': 'restock'}]


/var/folders/hl/w6fzrkk57tzdg3sb3bjn__zm0000gn/T/ipykernel_44721/3925791885.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  pd.DataFrame([event for event in LOG if event["gen_ai.conversation.id"] == restock_task]).fillna("")


,time,gen_ai.operation.name,gen_ai.conversation.id,status,step,tokens,ms,gen_ai.tool.name,arguments,attempts,key,replayed
0,1.793648e+09,invoke_agent,task_8ae044,started,,,,,,,,
1,1.793648e+09,chat,task_8ae044,,1.0,324.0,750.0,,,,,
2,1.793648e+09,execute_tool,task_8ae044,ok,1.0,,46.0,adjust_stock,"{'delta': 20, 'reason': 'restock', 'sku': 'SKU...",2.0,key_d7818f94d923e521,True
3,1.793648e+09,chat,task_8ae044,,2.0,361.0,509.0,,,,,
4,1.793648e+09,invoke_agent,task_8ae044,done,,,,,,,,


## 8. Cost and latency per task
This is the number to watch in production: what each task costs, not each call.

In [14]:
erp = AcmeERP()
run_agent(ask_model, TOOLS, "How many units of SKU-007 and SKU-008 do we have in total?")

events = pd.DataFrame(LOG)
chats = events[events["gen_ai.operation.name"] == "chat"]
tool_calls = events[events["gen_ai.operation.name"] == "execute_tool"]

per_task = pd.DataFrame({
    "model_calls": chats.groupby("gen_ai.conversation.id").size(),
    "tokens": chats.groupby("gen_ai.conversation.id")["tokens"].sum(),
    "model_ms": chats.groupby("gen_ai.conversation.id")["ms"].sum(),
    "tool_calls": tool_calls.groupby("gen_ai.conversation.id").size(),
    "attempts": tool_calls.groupby("gen_ai.conversation.id")["attempts"].sum(),
})
per_task.fillna(0).astype(int)

,model_calls,tokens,model_ms,tool_calls,attempts
gen_ai.conversation.id,,,,,
task_2b907d,3,1115,1152,2,2
task_8ae044,2,685,1259,1,2


## 9. Replaying a task offline
Something went wrong at 3 a.m. We run the same task again, feeding it the replies on the tape instead of
calling the model. Same decisions, against a clean ERP, at no cost.

In [ ]:
def replay(task_id):
    tape = TAPES[task_id]
    recorded = FakeModel(tape["replies"])
    return run_agent(recorded, TOOLS, tape["task"], task_id=task_id + "_replay")


def tools_called(task_id):
    calls = []
    for event in LOG:
        if event["gen_ai.conversation.id"] == task_id and event["gen_ai.operation.name"] == "execute_tool":
            calls.append((event["gen_ai.tool.name"], event["arguments"]))
    return calls


erp = AcmeERP()
answer, replay_task = replay(restock_task)
print("original:", tools_called(restock_task))
print("replay:  ", tools_called(replay_task))

Discussion: the key comes from the task and the arguments. When is that wrong? (Two identical writes that are both intended, in the same task.)

## 10. The harness matters more than the model
The model is a component you rent: the same weights are one API call away for anyone. What you *engineer* is
the harness around it — and under failure, the harness is what decides whether the data stays correct. To see
this, we hold the model's decisions fixed and change only the harness.

The decisions below are the ones a reasonable agent makes: restock `+20`, see a `502` (lost response), and
try again. We feed the *exact same* decisions to two harnesses:

- **naive** — call the tool, and on an error just call it again (no idempotency key),
- **robust** — the `execute` we built: a key per write, so a retry is deduplicated.

*Before you run it:* the first attempt commits then returns 502, so the agent retries. How many times does
each harness actually change the stock?

In [15]:
# The model's decisions, fixed: restock +20, then (after the 502) retry the same write.
DECISIONS = [("adjust_stock", {"sku": "SKU-012", "delta": 20, "reason": "restock"})] * 2


def harness_naive(erp, decisions):
    """No idempotency: on an error, blindly call the tool again."""
    for name, args in decisions:
        try:
            erp.adjust_stock(**args)
        except ERPError:
            try:
                erp.adjust_stock(**args)       # retry WITHOUT a key -> writes again
            except ERPError:
                pass


def harness_robust(erp, decisions, task_id):
    """Our execute(): one key per write, so the retry is deduplicated."""
    for name, args in decisions:
        execute(name, erp.adjust_stock, args, "write", task_id, sleep=no_wait)


ab_rows = []
for label, run in [("naive", harness_naive), ("robust", harness_robust)]:
    erp = AcmeERP()
    start = erp.products["SKU-012"]["stock"]
    erp.fail_next("lost_response")             # first attempt commits, then 502s
    if label == "naive":
        run(erp, DECISIONS)
    else:
        run(erp, DECISIONS, "restock_012")
    end = erp.products["SKU-012"]["stock"]
    ab_rows.append({"harness": label, "writes_applied": len(erp.movements),
                    "start": start, "end": end, "intended": start + 20,
                    "correct": end == start + 20})

pd.DataFrame(ab_rows)

,harness,writes_applied,start,end,intended,correct
0,naive,3,0,60,20,False
1,robust,1,0,20,20,True


In [16]:
# Now with a REAL weak model. gpt-oss-20b is 6x smaller than the 120b the lab uses by default,
# but run through the robust harness it handles the same lost-response task correctly.
# Needs network + a key; if the call fails (offline, rate limit) we skip without breaking the lab.
def ask_weak_model(messages, tools=None):
    kwargs = dict(model="openai/gpt-oss-20b", messages=messages, temperature=0)
    if tools:
        kwargs["tools"] = tools
    response = client.chat.completions.create(**kwargs)
    message = response.choices[0].message
    reply = {"role": "assistant", "content": message.content}
    if message.tool_calls:
        reply["tool_calls"] = [{"id": c.id, "type": "function",
                                "function": {"name": c.function.name, "arguments": c.function.arguments}}
                               for c in message.tool_calls]
    reply["tokens"] = response.usage.total_tokens
    return reply


try:
    erp = AcmeERP()
    start = erp.products["SKU-012"]["stock"]
    erp.fail_next("lost_response")                 # same infrastructure failure as above
    answer, weak_task = run_agent(ask_weak_model, TOOLS, "Restock SKU-012 with 20 units.")
    end = erp.products["SKU-012"]["stock"]
    print("weak model (gpt-oss-20b) + robust harness")
    print("  answer:", (answer or "").strip()[:100])
    print(f"  writes applied: {len(erp.movements)}  |  stock {start} -> {end}  |  correct: {end == start + 20}")
except Exception as error:
    print("Skipped the live weak-model call (no network / rate limited):", str(error)[:80])
    print("The deterministic A/B above already makes the point without the network.")

weak model (gpt-oss-20b) + robust harness
  answer: ✅ SKU-012 has been restocked with 20 units. The current stock is now 20 units.
  writes applied: 1  |  stock 0 -> 20  |  correct: True


Same decisions, same infrastructure failure. The naive harness applies the write twice and leaves the stock
wrong; the robust harness applies it once and is correct. The model never changed — only the code around it.

And the live run shows the other half: a model 6x smaller, run through the same robust harness, gets the
task right. A weaker model with a good harness beats a stronger model with a naive one, because under
failure the harness, not the model, is what keeps the data correct. This is why two products built on the
*same* API model can be worlds apart in reliability: the difference is engineered in the harness. Picking a
bigger model raises the ceiling on reasoning; engineering the harness raises the floor on correctness — and
in production it is the floor that gets you paged.

## 11. Harness engineering: the knobs are decisions, not constants
Every number in this notebook is a choice with a cost on each side. `keep_last` and `max_chars` (section 6)
trade **tokens we pay for** against **information the model still sees**: shorten too much and the model
loses a fact it needs; shorten too little and every step costs more. Engineering the harness means picking
these by measuring, not by taste.

The loop below runs the *same* conversation through several settings and tabulates the trade-off. We also
check whether a fact the task depends on (the stock of `SKU-012`) survives the shortening. No model calls:
we rebuild the catalog-review conversation from section 6, so this is instant and free.

*Before you run it:* which setting costs the fewest tokens? Is it the one you would ship?

In [17]:
# Rebuild a long conversation (same shape as section 6), self-contained so this cell runs on its own.
erp = AcmeERP()
needed_fact = f"SKU-012"            # a fact a later step might depend on
convo = [{"role": "system", "content": "..."},
         {"role": "user", "content": "Review the whole catalog, then tell me the stock of SKU-012"}]
for _ in range(6):
    reply = tool_reply("find_products")
    reply.pop("tokens")
    convo.append(reply)
    convo.append({"role": "tool", "tool_call_id": reply["tool_calls"][0]["id"],
                  "content": json.dumps(erp.find_products())})      # every result mentions SKU-012


def fact_survives(messages, fact):
    """Is the needed fact still present in any OLDER (shortened) tool result?"""
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    older = tool_msgs[:-1] if len(tool_msgs) > 1 else []
    return any(fact in m["content"] for m in older)


full_tokens = approx_tokens(convo)
settings = [
    {"keep_last": 2, "max_chars": 120},     # the notebook's default
    {"keep_last": 1, "max_chars": 60},      # aggressive: cheapest
    {"keep_last": 2, "max_chars": 400},     # roomy: keeps more text
    {"keep_last": 6, "max_chars": 120},     # keep every result in full
]

rows = []
for s in settings:
    shortened = shorten_old_results(convo, keep_last=s["keep_last"], max_chars=s["max_chars"])
    tokens = approx_tokens(shortened)
    rows.append({
        **s,
        "tokens": tokens,
        "saved_%": round(100 * (full_tokens - tokens) / full_tokens),
        "fact_kept": fact_survives(shortened, needed_fact),
    })

print("full context:", full_tokens, "tokens\n")
pd.DataFrame(rows)

full context: 2440 tokens



,keep_last,max_chars,tokens,saved_%,fact_kept
0,2,120,1212,50,True
1,1,60,817,67,False
2,2,400,1535,37,True
3,6,120,2440,0,True


Reading the table is the job. The cheapest row (`keep_last=1, max_chars=60`) drops `fact_kept` to `False`:
it saves the most tokens *and* loses the stock number the task still needs — a bug that shows up as a wrong
answer, not an error. The default row keeps the fact at a fair cost; the roomy row keeps it for a few more
tokens; the last row keeps everything and saves almost nothing on this short run but is the safest as tasks
grow. **The engineered choice is the smallest context in which `fact_kept` stays `True`** — and the only way
to know that line is to measure it, which is exactly what a harness with a log lets you do.

This is the loop for every knob in the notebook — `max_attempts`, `cooldown_seconds`, `keep_last`: pick a
value, run it against cases you care about, read the cost **and** the failures, then move the knob. That loop,
not any single number, is harness engineering.

## Take away
- A write is only safe to retry if the system honours an idempotency key.
- Errors go back to the model with a hint. 'Not found' and 'forbidden' are never retried.
- Shorten old results but keep the structure. Long-term memory is an attack surface.
- Log metadata with standard names; keep the model's replies on a separate tape to replay.
- Under failure the harness, not the model, keeps the data correct: a weak model with a good harness beats a strong model with a naive one.
- Every knob is a decision: tune it by measuring cost **and** the failures it causes, never by taste.